In [ ]:
# ============================================================================
# ETAPA 10 — MODELO DE RISCOS COMPETITIVOS (P→S + P→N)
# ============================================================================
# Integra os dois processos num modelo conjunto:
#   P→S: conversão para soja (risco de perda)
#   P→N: regeneração para nativa (risco de ganho)
#
# Framework de riscos competitivos:
#   Cada pixel pode sofrer um de dois eventos, e a ocorrência
#   de um impede o outro. O modelo estima:
#
#   h_S(t): taxa de risco instantânea de conversão para soja
#   h_N(t): taxa de risco instantânea de regeneração
#   h_total(t) = h_S(t) + h_N(t)
#
#   S_total(t) = exp(-integral[h_total]) = S_S(t) * S_N(t)
#
#   CIF_S(t) = P(converter para soja em ≤t, antes de regenerar)
#   CIF_N(t) = P(regenerar em ≤t, antes de converter)
#
# A retroalimentação entre os dois processos emerge naturalmente:
#   Pixels com h_S alto têm S_total que cai rapidamente,
#   reduzindo a janela disponível para regeneração.
#
# Arquitetura:
#   Shared Encoder (290→256→128)
#   Head P→S: (128→64→2) → (log_k_S, log_lam_S)
#   Head P→N: (128→64→2) → (log_k_N, log_lam_N)
#
# Loss combinada:
#   L = L_S + L_N
#   L_S: log-lik Weibull para evento P→S
#   L_N: log-lik Weibull para evento P→N (com class weight)
# ============================================================================

import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm
import json
from datetime import datetime
from sklearn.metrics import roc_auc_score

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED   = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Configuração
ANO_FIM        = 2024
HORIZONTE_MAX  = 10    # janela máxima de observação
WEIGHT_N       = 5.6   # class weight para P→N (regeneração rara)
ALPHA          = 0.5   # peso relativo L_S vs L_N (0.5 = igual)

# Horizontes de avaliação
HORIZONS = [2, 5, 10]

print(f"✅ Configuração Riscos Competitivos")
print(f"   Device:    {DEVICE}")
print(f"   Processos: P→S (conversão) + P→N (regeneração)")
print(f"   Loss:      alpha={ALPHA} L_S + (1-alpha)={1-ALPHA} L_N")
print(f"   Weight P→N: {WEIGHT_N}x")
print(f"   Horizonte:  {HORIZONTE_MAX} anos")

In [ ]:
# ============================================================================
# MODELO DE RISCOS COMPETITIVOS
# ============================================================================

class CompetingRisksModel(nn.Module):
    """
    Modelo Weibull de riscos competitivos P→S e P→N.

    Encoder compartilhado aprende representação comum.
    Duas heads independentes estimam (k, lambda) por processo.

    Input:  features (290,)
    Output: (log_k_S, log_lam_S, log_k_N, log_lam_N)
    """

    def __init__(self, input_dim=290, hidden_dim=256,
                 encoder_dim=128, head_hidden=64, dropout=0.3):
        super().__init__()

        # Encoder compartilhado
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, encoder_dim), nn.ReLU(), nn.Dropout(dropout)
        )

        # Head P→S (conversão para soja)
        self.head_S = nn.Sequential(
            nn.Linear(encoder_dim, head_hidden), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(head_hidden, 2)
        )

        # Head P→N (regeneração)
        self.head_N = nn.Sequential(
            nn.Linear(encoder_dim, head_hidden), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(head_hidden, 2)
        )

        # Inicialização baseada nos modelos individuais
        # P→S: k≈1.9, lambda≈15 | P→N: k≈4.0, lambda≈37
        with torch.no_grad():
            self.head_S[-1].bias[0] = np.log(1.9)   # log_k_S
            self.head_S[-1].bias[1] = np.log(15.0)  # log_lam_S
            self.head_N[-1].bias[0] = np.log(4.0)   # log_k_N
            self.head_N[-1].bias[1] = np.log(37.0)  # log_lam_N

        # Limites físicos
        self.log_k_min   = np.log(0.1);  self.log_k_max   = np.log(5.0)
        self.log_lam_min = np.log(0.5);  self.log_lam_max = np.log(50.0)

    def forward(self, features):
        enc = self.encoder(features)

        wb_S = self.head_S(enc)
        wb_N = self.head_N(enc)

        log_k_S   = wb_S[:,0].clamp(self.log_k_min, self.log_k_max)
        log_lam_S = wb_S[:,1].clamp(self.log_lam_min, self.log_lam_max)
        log_k_N   = wb_N[:,0].clamp(self.log_k_min, self.log_k_max)
        log_lam_N = wb_N[:,1].clamp(self.log_lam_min, self.log_lam_max)

        return log_k_S, log_lam_S, log_k_N, log_lam_N

    def cif(self, features, t_vals):
        """
        Cumulative Incidence Functions (CIF) para ambos os processos.

        CIF_S(t) = integral_0^t h_S(u) * S_total(u) du
        Para Weibull com dois riscos competitivos:
          S_total(t) = S_S(t) * S_N(t)
          CIF_S(t) ≈ 1 - S_S(t)  (quando os dois processos são independentes)

        Retorna: (CIF_S, CIF_N) por pixel para cada t em t_vals
        """
        log_k_S, log_lam_S, log_k_N, log_lam_N = self.forward(features)
        k_S = torch.exp(log_k_S); lam_S = torch.exp(log_lam_S)
        k_N = torch.exp(log_k_N); lam_N = torch.exp(log_lam_N)

        results = []
        for t in t_vals:
            S_S = torch.exp(-(t/lam_S)**k_S)   # sobrevivência P→S
            S_N = torch.exp(-(t/lam_N)**k_N)   # sobrevivência P→N
            S_T = S_S * S_N                    # sobrevivência total
            # CIF aproximada (independência entre riscos)
            CIF_S = 1 - S_S
            CIF_N = 1 - S_N
            results.append((CIF_S, CIF_N, S_T))
        return results


def competing_risks_loss(log_k_S, log_lam_S, log_k_N, log_lam_N,
                         times_S, events_S, times_N, events_N,
                         alpha=ALPHA, weight_N=WEIGHT_N):
    """
    Loss combinada para riscos competitivos.
    alpha: peso de L_S (0.5 = igual importância)
    """
    eps = 1e-6

    def weibull_ll(log_k, log_lam, times, events, weight=1.0):
        k   = torch.exp(log_k)   + eps
        lam = torch.exp(log_lam) + eps
        t   = times.clamp(min=eps)
        ev  = events.float()
        log_f = (torch.log(k) - torch.log(lam) +
                 (k-1)*(torch.log(t)-torch.log(lam)) - (t/lam)**k)
        log_S = -((t/lam)**k)
        w = 1.0 + ev * (weight - 1.0)
        return -(w * (ev*log_f + (1-ev)*log_S)).mean()

    L_S = weibull_ll(log_k_S, log_lam_S, times_S, events_S, weight=1.0)
    L_N = weibull_ll(log_k_N, log_lam_N, times_N, events_N, weight=weight_N)

    return alpha * L_S + (1-alpha) * L_N, L_S, L_N


# Verificação
torch.manual_seed(SEED)
model = CompetingRisksModel().to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())

dummy = torch.randn(4, 290).to(DEVICE)
lk_S, ll_S, lk_N, ll_N = model(dummy)

print(f"✅ CompetingRisksModel criado!")
print(f"   Parâmetros: {total_params:,}")
print(f"   k_S inicial: {torch.exp(lk_S).mean().item():.2f}  "
      f"(esperado ≈ 1.9)")
print(f"   k_N inicial: {torch.exp(lk_N).mean().item():.2f}  "
      f"(esperado ≈ 4.0)")
print(f"   lam_S:  {torch.exp(ll_S).mean().item():.1f} anos")
print(f"   lam_N:  {torch.exp(ll_N).mean().item():.1f} anos")

In [ ]:
# ============================================================================
# DATASET DE RISCOS COMPETITIVOS
# ============================================================================
# Cada pixel tem DOIS tempos e DOIS eventos:
#   (tempo_S, evento_S): tempo até conversão para soja
#   (tempo_N, evento_N): tempo até regeneração para nativa
#
# Para um pixel que converteu para soja:
#   evento_S=1, tempo_S=anos até soja
#   evento_N=0, tempo_N=min(ANO_FIM-T, HORIZONTE_MAX) [censurado]
#
# Para um pixel que regenerou:
#   evento_S=0, tempo_S=min(ANO_FIM-T, HORIZONTE_MAX) [censurado]
#   evento_N=1, tempo_N=anos até regeneração
#
# Para um pixel que não fez nenhuma transição:
#   evento_S=0, evento_N=0 (duplamente censurado)

import rasterio
from rasterio.windows import Window

DATA_DIR    = r"D:\Projetos\Cerrado\LULC"
PATTERN     = "brazil_coverage_{ano}_Cerrado.tif"
YEARS       = list(range(1985, 2025))
NODATA_IN   = 255; NODATA_OUT = 0
PATCH_N     = 7;   MAX_SERIE  = len(YEARS)-1;  PATCH_YEARS = 5

def _ler_pixel(year, row, col):
    with rasterio.open(os.path.join(
            DATA_DIR, PATTERN.format(ano=year))) as ds:
        return int(ds.read(1, window=Window(col,row,1,1),
                           out_dtype="uint8")[0,0])

def _ler_patch(year, row, col):
    h = PATCH_N//2
    with rasterio.open(os.path.join(
            DATA_DIR, PATTERN.format(ano=year))) as ds:
        H,W = ds.height, ds.width
        c0 = min(max(0,col-h),W-PATCH_N)
        r0 = min(max(0,row-h),H-PATCH_N)
        arr = ds.read(1, window=Window(c0,r0,PATCH_N,PATCH_N),
                      out_dtype="uint8")
    return np.where(arr==NODATA_IN,NODATA_OUT,arr).astype(np.float32)


class CompetingRisksDataset(Dataset):
    """
    Dataset para riscos competitivos.
    Une pixels do dataset P→S e P→N com o mesmo split espacial.
    Retorna: features, tempo_S, evento_S, tempo_N, evento_N
    """

    def __init__(self, df):
        self.df = df.reset_index(drop=True)
        n_S = self.df['evento_S'].sum()
        n_N = self.df['evento_N'].sum()
        n_both = ((self.df['evento_S']==0) & (self.df['evento_N']==0)).sum()
        print(f"  Dataset: {len(self.df):,} pixels")
        print(f"  Evento P→S (soja):       {int(n_S):,}")
        print(f"  Evento P→N (regeneração): {int(n_N):,}")
        print(f"  Duplamente censurados:   {int(n_both):,}")

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        r      = self.df.iloc[idx]
        row    = int(r['row']); col = int(r['col'])
        T      = int(r['T'])
        t_S    = float(r['tempo_S']); ev_S = int(r['evento_S'])
        t_N    = float(r['tempo_N']); ev_N = int(r['evento_N'])
        grupo  = str(r.get('grupo_T', 'NC'))

        anos_s = [y for y in YEARS if y < T]
        if anos_s:
            sr = np.array([_ler_pixel(y,row,col)
                           for y in anos_s], dtype=np.float32)
            sr = np.where(sr==NODATA_IN,NODATA_OUT,sr).astype(np.float32)
        else:
            sr = np.array([], dtype=np.float32)
        sl = len(sr)
        serie = np.zeros(MAX_SERIE, dtype=np.float32)
        if sl > 0: serie[MAX_SERIE-sl:] = sr

        anos_p = [y for y in YEARS if y<T][-PATCH_YEARS:]
        patch  = np.zeros((PATCH_YEARS,PATCH_N,PATCH_N), dtype=np.float32)
        for i,y in enumerate(anos_p):
            patch[PATCH_YEARS-len(anos_p)+i] = _ler_patch(y,row,col)

        has21 = float(np.sum(sr==21)/sl) if sl>0 else 0.0
        apast = sum(1 for v in reversed(sr) if v==15) if sl>0 else 0
        ctm1  = float(sr[-1]) if sl>0 else 0.0
        aux   = np.array([has21,float(apast),ctm1], dtype=np.float32)

        isCC = 1.0 if grupo=='CC* (cluster)'  else 0.0
        isCD = 1.0 if grupo=='CD* (disperso)' else 0.0
        isNC = 1.0 if grupo=='NC'             else 0.0
        ctx  = np.array([isCC,isCD,isNC], dtype=np.float32)

        features = np.concatenate([serie, patch.flatten(), aux, ctx])
        assert features.shape == (290,)

        return (
            torch.tensor(features, dtype=torch.float32),
            torch.tensor(t_S,  dtype=torch.float32),
            torch.tensor(ev_S, dtype=torch.float32),
            torch.tensor(t_N,  dtype=torch.float32),
            torch.tensor(ev_N, dtype=torch.float32)
        )


# Construir dataset conjunto
BASE_DIR = Path(r"D:\Projetos\Cerrado\GeoFM_sampling")
DATA_V2  = BASE_DIR / "dataset_v2"
DATA_PN  = BASE_DIR / "dataset_PN"

def build_competing_dataset(split):
    """Une P→S e P→N para o mesmo split espacial."""

    # P→S
    df_S = pd.read_csv(DATA_V2 / f"dataset_v2_{split}.csv")
    df_S = df_S[df_S['label'].notna()].copy()
    df_S['label'] = df_S['label'].astype(int)
    df_S['T']     = df_S['T'].astype(int)

    # Recuperar ano_conversao
    df_S_full = pd.read_csv(DATA_V2 / "dataset_v2_full.csv")
    df_ck     = pd.read_csv(DATA_V2 / "recalc_checkpoint.csv")
    df_S_full = df_S_full.merge(
        df_ck[['row','col','T','ano_conversao']].drop_duplicates(),
        on=['row','col','T'], how='left')
    df_S = df_S.merge(
        df_S_full[['row','col','T','ano_conversao','grupo_T']].drop_duplicates(),
        on=['row','col','T'], how='left')

    df_S['tempo_S'] = np.where(
        df_S['label']==1,
        (df_S['ano_conversao']-df_S['T']).clip(1, HORIZONTE_MAX),
        np.minimum(ANO_FIM-df_S['T'], HORIZONTE_MAX))
    df_S['evento_S'] = df_S['label']

    # P→N
    df_N = pd.read_csv(DATA_PN / f"dataset_PN_{split}.csv")
    df_N = df_N[df_N['label'].notna()].copy()
    df_N['label'] = df_N['label'].astype(int)
    df_N['T']     = df_N['T'].astype(int)

    df_N_full = pd.read_csv(DATA_PN / "dataset_PN_full.csv")
    df_N = df_N.merge(
        df_N_full[['row','col','T','ano_regeneracao']].drop_duplicates(),
        on=['row','col','T'], how='left')
    df_N['tempo_N'] = np.where(
        df_N['label']==1,
        (df_N['ano_regeneracao']-df_N['T']).clip(1, HORIZONTE_MAX),
        np.minimum(ANO_FIM-df_N['T'], HORIZONTE_MAX))
    df_N['evento_N'] = df_N['label']

    # Unir pelos pixels comuns
    df_N_slim = df_N[['row','col','T','tempo_N','evento_N']]
    df = df_S.merge(df_N_slim, on=['row','col','T'], how='left')

    # Pixels só em P→S (sem match P→N) → evento_N=0, censurado
    df['tempo_N']  = df['tempo_N'].fillna(
        np.minimum(ANO_FIM-df['T'], HORIZONTE_MAX))
    df['evento_N'] = df['evento_N'].fillna(0).astype(int)
    df['grupo_T']  = df['grupo_T'].fillna('NC')

    df = df[df['tempo_S']>0].copy()
    return df


print("Construindo datasets de riscos competitivos...")
print()
print("Train:"); df_train = build_competing_dataset('train')
train_ds = CompetingRisksDataset(df_train)
print()
print("Val:");   df_val   = build_competing_dataset('val')
val_ds   = CompetingRisksDataset(df_val)
print()
print("Test:");  df_test  = build_competing_dataset('test')
test_ds  = CompetingRisksDataset(df_test)

BATCH_SIZE = 256
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=0)
print(f"\n✅ DataLoaders prontos!")

In [ ]:
# ============================================================================
# TREINO
# ============================================================================

def train_epoch(model, loader, optimizer, device):
    model.train()
    tot_loss = tot_S = tot_N = n = 0
    k_S_vals = []; k_N_vals = []

    pbar = tqdm(loader, desc="Training")
    for feat, t_S, ev_S, t_N, ev_N in pbar:
        feat = feat.to(device)
        t_S  = t_S.to(device);  ev_S = ev_S.to(device)
        t_N  = t_N.to(device);  ev_N = ev_N.to(device)

        lk_S, ll_S, lk_N, ll_N = model(feat)
        loss, L_S, L_N = competing_risks_loss(
            lk_S, ll_S, lk_N, ll_N, t_S, ev_S, t_N, ev_N)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        tot_loss += loss.item(); tot_S += L_S.item()
        tot_N += L_N.item(); n += 1
        k_S_vals.extend(torch.exp(lk_S).detach().cpu().numpy())
        k_N_vals.extend(torch.exp(lk_N).detach().cpu().numpy())

        pbar.set_postfix({
            'L':   f'{loss.item():.3f}',
            'L_S': f'{L_S.item():.3f}',
            'L_N': f'{L_N.item():.3f}'
        })

    return (tot_loss/n, tot_S/n, tot_N/n,
            float(np.mean(k_S_vals)), float(np.mean(k_N_vals)))


def evaluate(model, loader, device, collect=False):
    model.eval()
    tot_loss = n = 0
    all_pS, all_pN, all_evS, all_evN = [], [], [], []
    all_kS, all_kN, all_lS, all_lN   = [], [], [], []

    with torch.no_grad():
        for feat, t_S, ev_S, t_N, ev_N in tqdm(loader, desc="Eval"):
            feat = feat.to(device)
            t_S  = t_S.to(device);  ev_S = ev_S.to(device)
            t_N  = t_N.to(device);  ev_N = ev_N.to(device)

            lk_S, ll_S, lk_N, ll_N = model(feat)
            loss, _, _ = competing_risks_loss(
                lk_S, ll_S, lk_N, ll_N, t_S, ev_S, t_N, ev_N)
            tot_loss += loss.item(); n += 1

            k_S = torch.exp(lk_S); lam_S = torch.exp(ll_S)
            k_N = torch.exp(lk_N); lam_N = torch.exp(ll_N)

            # CIF em t=10 anos
            t = torch.tensor(10.0).to(device)
            p_S = 1 - torch.exp(-(t/lam_S)**k_S)
            p_N = 1 - torch.exp(-(t/lam_N)**k_N)

            all_pS.extend(p_S.cpu().numpy())
            all_pN.extend(p_N.cpu().numpy())
            all_evS.extend(ev_S.cpu().numpy())
            all_evN.extend(ev_N.cpu().numpy())
            if collect:
                all_kS.extend(k_S.cpu().numpy())
                all_kN.extend(k_N.cpu().numpy())
                all_lS.extend(lam_S.cpu().numpy())
                all_lN.extend(lam_N.cpu().numpy())

    pS = np.array(all_pS); pN = np.array(all_pN)
    eS = np.array(all_evS); eN = np.array(all_evN)

    try:    auc_S = roc_auc_score(eS, pS)
    except: auc_S = 0.0
    try:    auc_N = roc_auc_score(eN, pN)
    except: auc_N = 0.0

    metrics = {'loss': tot_loss/n, 'auc_S': float(auc_S), 'auc_N': float(auc_N)}
    extras  = (np.array(all_kS), np.array(all_kN),
               np.array(all_lS), np.array(all_lN),
               pS, pN, eS, eN) if collect else None
    return metrics, extras


# Treino
torch.manual_seed(SEED)
model     = CompetingRisksModel().to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=5, factor=0.5, verbose=True)

N_EPOCHS = 30; EARLY_STOP_PAT = 10
best_val_loss = float('inf'); best_model_state = None
best_val_metrics = None; patience_counter = 0
history = []

print("=" * 70)
print("TREINO — CompetingRisksModel")
print("=" * 70)
print(f"  Train: {len(train_ds):,}  Val: {len(val_ds):,}  Test: {len(test_ds):,}")
print(f"  Loss: alpha={ALPHA}*L_S + {1-ALPHA}*L_N")
print(f"  Referências:")
print(f"    Weibull P→S (etapa8): AUC=0.846")
print(f"    Weibull P→N (etapa9): AUC=0.910")
print()

for epoch in range(N_EPOCHS):
    print(f"Epoch {epoch+1}/{N_EPOCHS}")
    tl, ts, tn, kS, kN = train_epoch(model, train_loader, optimizer, DEVICE)
    val_m, _ = evaluate(model, val_loader, DEVICE)
    scheduler.step(val_m['loss'])
    history.append({'train_loss':tl,'L_S':ts,'L_N':tn,**val_m})

    print(f"Train - L={tl:.4f}  L_S={ts:.4f}  L_N={tn:.4f}  "
          f"k_S={kS:.2f}  k_N={kN:.2f}")
    print(f"Val   - Loss={val_m['loss']:.4f}  "
          f"AUC_S={val_m['auc_S']:.4f}  AUC_N={val_m['auc_N']:.4f}")

    if val_m['loss'] < best_val_loss:
        best_val_loss    = val_m['loss']
        best_model_state = {k: v.cpu().clone()
                            for k, v in model.state_dict().items()}
        best_val_metrics = val_m.copy()
        patience_counter = 0
        print("  → Best model updated!")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PAT:
            print(f"\nEarly stopping após {epoch+1} epochs.")
            break
    print()

print(f"Melhor val AUC_S: {best_val_metrics['auc_S']:.4f}")
print(f"Melhor val AUC_N: {best_val_metrics['auc_N']:.4f}")

In [ ]:
# ============================================================================
# AVALIAÇÃO + VISUALIZAÇÃO DA RETROALIMENTAÇÃO + FREEZE
# ============================================================================

import matplotlib.pyplot as plt

model.load_state_dict(best_model_state)
model.to(DEVICE)

test_metrics, (kS_np,kN_np,lS_np,lN_np,pS_np,pN_np,eS_np,eN_np) = evaluate(
    model, test_loader, DEVICE, collect=True)

print("=" * 70)
print("AVALIAÇÃO FINAL — TEST SET")
print("=" * 70)
print(f"  AUC P→S: {test_metrics['auc_S']:.4f}  "
      f"(etapa8 individual: 0.846)")
print(f"  AUC P→N: {test_metrics['auc_N']:.4f}  "
      f"(etapa9 individual: 0.910)")
print(f"\nParâmetros Weibull (test set):")
print(f"  k_S: mean={kS_np.mean():.3f}  std={kS_np.std():.3f}")
print(f"  k_N: mean={kN_np.mean():.3f}  std={kN_np.std():.3f}")
print(f"  lam_S: mean={lS_np.mean():.2f}  lam_N: mean={lN_np.mean():.2f}")

# Retroalimentação: correlação P→S vs P→N por pixel
corr = np.corrcoef(pS_np, pN_np)[0,1]
print(f"\nRetroalimentação (correlação P_S vs P_N por pixel):")
print(f"  r={corr:.3f} "
      f"({'✅ negativa — riscos competitivos' if corr < -0.1 else '⚠️  fraca'})"
      f"\n  (esperado: negativa — alto risco de conversão implica "
      f"baixo risco de regeneração)")

# Gráfico 4 painéis
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# 1. Curvas P→S vs P→N médias (competing risks)
t_plot = np.linspace(0.1, 25, 300)
kS_m=kS_np.mean(); lS_m=lS_np.mean()
kN_m=kN_np.mean(); lN_m=lN_np.mean()
CIF_S = 1 - np.exp(-(t_plot/lS_m)**kS_m)
CIF_N = 1 - np.exp(-(t_plot/lN_m)**kN_m)
S_T   = np.exp(-(t_plot/lS_m)**kS_m) * np.exp(-(t_plot/lN_m)**kN_m)
axes[0,0].plot(t_plot, CIF_S, '#c0392b', lw=2.5, label='CIF P→S (conversão)')
axes[0,0].plot(t_plot, CIF_N, '#27ae60', lw=2.5, label='CIF P→N (regeneração)')
axes[0,0].plot(t_plot, 1-S_T, '#8e44ad', lw=2, linestyle='--',
               label='P(qualquer evento)')
axes[0,0].fill_between(t_plot, CIF_S, CIF_S+CIF_N,
                        alpha=0.15, color='#27ae60')
axes[0,0].set_xlabel('Anos em pastagem')
axes[0,0].set_ylabel('P(evento ≤t anos)')
axes[0,0].set_title('CIF: Riscos Competitivos', fontweight='bold')
axes[0,0].legend(fontsize=9)
axes[0,0].set_xlim(0,25); axes[0,0].set_ylim(0,1)

# 2. Scatter P_S vs P_N por pixel (retroalimentação)
axes[0,1].scatter(pS_np, pN_np, alpha=0.15, s=8, c='steelblue')
axes[0,1].set_xlabel('P(P→S em ≤10 anos)')
axes[0,1].set_ylabel('P(P→N em ≤10 anos)')
axes[0,1].set_title(f'Retroalimentação por pixel\n(r={corr:.3f})',
                     fontweight='bold')
axes[0,1].set_xlim(0,1); axes[0,1].set_ylim(0,1)
axes[0,1].plot([0,1],[0,1],'k--',alpha=0.3)

# 3. Distribuição de k_S e k_N
axes[1,0].hist(kS_np, bins=30, alpha=0.6, color='#c0392b',
               density=True, label=f'k_S P→S (mean={kS_np.mean():.2f})')
axes[1,0].hist(kN_np, bins=30, alpha=0.6, color='#27ae60',
               density=True, label=f'k_N P→N (mean={kN_np.mean():.2f})')
axes[1,0].axvline(x=1, color='black', linestyle='--', alpha=0.5)
axes[1,0].set_xlabel('k (forma Weibull)')
axes[1,0].set_ylabel('Densidade')
axes[1,0].set_title('k por processo', fontweight='bold')
axes[1,0].legend()

# 4. Comparação AUC: individual vs competitivo
modelos = ['Weibull P→S\n(individual)', 'Weibull P→N\n(individual)',
           'Competing\nP→S', 'Competing\nP→N']
aucs    = [0.846, 0.910,
           test_metrics['auc_S'], test_metrics['auc_N']]
cores   = ['#c0392b','#27ae60','#c0392b','#27ae60']
alphas  = [0.5, 0.5, 1.0, 1.0]
bars = axes[1,1].bar(modelos, aucs, color=cores,
                      alpha=0.8, edgecolor='white')
for bar, auc in zip(bars, aucs):
    axes[1,1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                   f'{auc:.3f}', ha='center', fontsize=10, fontweight='bold')
axes[1,1].set_ylim(0.8, 1.0)
axes[1,1].set_ylabel('AUC')
axes[1,1].set_title('AUC: individual vs competitivo', fontweight='bold')
axes[1,1].axhline(y=0.846, color='#c0392b', linestyle=':', alpha=0.5)
axes[1,1].axhline(y=0.910, color='#27ae60', linestyle=':', alpha=0.5)

plt.suptitle('Modelo de Riscos Competitivos P→S + P→N',
             fontsize=13, fontweight='bold')
plt.tight_layout()

# Freeze
OUT_DIR   = BASE_DIR / "etapa10_frozen"
OUT_DIR.mkdir(exist_ok=True, parents=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

plt.savefig(str(OUT_DIR / f'competing_risks_{timestamp}.png'),
            dpi=150, bbox_inches='tight')
plt.show()

checkpoint = {
    'model_state_dict': best_model_state,
    'model_class': 'CompetingRisksModel',
    'config': {'input_dim':290,'hidden_dim':256,
               'encoder_dim':128,'head_hidden':64,'dropout':0.3},
    'transitions': ['P_to_S','P_to_N'],
    'metrics': {
        'validation': {k: float(v) for k,v in best_val_metrics.items()},
        'test':       {k: float(v) for k,v in test_metrics.items()}
    },
    'weibull_params': {
        'k_S_mean': float(kS_np.mean()), 'k_S_std': float(kS_np.std()),
        'k_N_mean': float(kN_np.mean()), 'k_N_std': float(kN_np.std()),
        'lam_S_mean': float(lS_np.mean()), 'lam_N_mean': float(lN_np.mean())
    },
    'feedback': {'corr_pS_pN': float(corr)},
    'training': {'alpha':ALPHA,'weight_N':WEIGHT_N,
                 'n_epochs':len(history),'seed':SEED}
}
pth = OUT_DIR / f"competing_risks_{timestamp}.pth"
torch.save(checkpoint, pth)

results = {
    'model': 'CompetingRisksModel',
    'transitions': ['P_to_S','P_to_N'],
    'metrics': checkpoint['metrics'],
    'weibull_params': checkpoint['weibull_params'],
    'feedback_corr': float(corr),
    'comparison': {
        'etapa8_auc_S': 0.846,
        'etapa9_auc_N': 0.910,
        'etapa10_auc_S': float(test_metrics['auc_S']),
        'etapa10_auc_N': float(test_metrics['auc_N'])
    }
}
with open(OUT_DIR/f"results_{timestamp}.json",'w',encoding='utf-8') as f:
    json.dump(results, f, indent=2)

print(f"\n✅ CompetingRisksModel congelado: {pth.name}")
print(f"   {OUT_DIR}")